# 28 — Single-feature BEDROC

## A case study in the fragility of one-feature virtual-screening claims

This notebook documents the retraction and hardening of Claim B from the first pass. The original claim:

<!-- canonical baseline — see data/derived/canonical_baselines.csv -->

> `lig_buried_sasa_std_A2` used as a single-feature ranker achieves panel BEDROC α=20 = 0.674, beating the GBSA-locked baseline of 0.609. This is an induced-fit signature.

(The 0.674 vs 0.609 figures are both on the 8-target subset that silently dropped 4A5S. On the canonical 9-target panel with 4A5S imputed at 0, `lig_buried_sasa_std_A2` scores 0.603 [0.39, 0.78] vs canonical GBSA-locked 0.541 — a point delta of +0.062, well inside the GBSA CI, and the feature collapses under MW-residualisation and PBC-cleaning. See `data/derived/canonical_baselines.csv`. Two sum-of-partial-charge columns — `lig_partial_q_sum` in `ligand_chem.parquet` and `ligand_partial_charge_sum` in `features.parquet` — are the same neutral-by-construction column (all 270 values essentially zero) and produced spurious BEDROC values from roundoff noise; both are excluded from the ranking. See `docs/GLOSSARY.md`.)

Five independent reviewers took the original claim apart on four lines of attack: non-reproducibility, molecular-weight confounding, collapse under bound-only trajectories, and decoy-label incoherence. The claim is retracted in its original form. A residual signal survives only under a very tight active/decoy definition and needs external validation before it can be used for screening.

The rest of this notebook walks through the naive positive result and the hardened comparison side by side, and lands on an explicit verdict.

_(Self-contained: loads its data via `discovery9.io`, exports figures to `figures/28_single_feature_bedroc_figK.png`.)_

> **Bootstrap regime.** This notebook uses B=1000, seed 20250901 for speed. The canonical repo-wide regime (see `data/derived/canonical_baselines.csv`) is B=5000, seed 20260902. Numbers here are stable at reported precision; CIs are a hair wider.

> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-feature analysis and downstream ranking questions.
>
> See STUDY_DESIGN Chapter §A3 Q1 (per-target combo selection) and Q2 (single-feature panel
> ranker) for the framing this notebook addresses.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` +
> `data/raw/reference/ohds_metadata.csv` (and `data/derived/canonical_baselines.csv` for
> baseline comparison).

> **Reader's guide — what this notebook does, in plain language**\n>\n> **Question:** Can a single MD descriptor (e.g. \"fluctuation of buried SASA\")\n> rank actives better than GBSA?\n>\n> **Why that would be interesting:** 30 ns MD runs, one number per complex → BEDROC.\n> No fitting, no CV, no ensembling. If a single feature manages this = 1000× cheaper\n> than a GBSA rescore.\n>\n> **The original \"Claim B\":** `lig_buried_sasa_std_A2` (the std of buried SASA\n> along the trajectory) gave panel BEDROC = **0.674** vs GBSA-locked 0.609.\n> Interpreted as an \"induced-fit signature\": the pocket rearranges around actives,\n> giving wide SASA fluctuation.\n>\n> **Method:** For every feature (60+ MD + 10 ligand-chem):\n> 1. Compute per-target BEDROC (no ML, just ranking)\n> 2. Try both directions (+ and −), take argmax\n> 3. Panel mean = 8 target values averaged (4A5S excluded, no GBSA labels)\n>\n> **Then the 4 hardening attacks — 7 conditions total:**\n>\n> | Condition | What it does | BEDROC |\n> |---|---|---:|\n> | **naive** | Original claim, 8T subset | 0.674 |\n> | **canonical 9T** (4A5S imputed=0) | Honest baseline | 0.603 vs GBSA 0.541 |\n> | **mw_resid** | Score residualised on ligand MW per target | **0.506** (⬇ below GBSA!) |\n> | **size_resid** | Residualised on n_atoms | similar |\n> | **bound_only** | Only trajectories where ligand still bound at end | **0.463** (⬇ below random!) |\n> | **pbc_clean** | PBC-artefact trajectories excluded | drops |\n> | **alt_label_7** | Tight labels (active pchembl≥7, decoy≤5, middle dropped) | 0.978 p=0.015 **BUT** 3/8 targets have 0 decoys → degenerate panel |\n>\n> **Statistics:** Bootstrap 95 % CI with B=5000 resamples (targets with replacement),\n> permutation p with B=5000.\n>\n> **How to read the numbers:**\n> - **Δ vs GBSA CI contains 0** → signal statistically indistinguishable from noise.\n> - **Value drops below GBSA after hardening** → the naive value was a confounder\n>   (here MW confound).\n> - **alt_label_7 = 0.978**: sounds spectacular, but degenerate panel (3 targets\n>   without decoys get silently dropped → only 5 targets count, of which just 1 evaluable).\n>\n> **Pedagogical lesson:** A feature that beats a specific cutoff under a specific\n> panel definition isn't a discovery — it's an **artefact of a specific analysis\n> convention**. Always test under perturbations (label cutoff, MW residual, panel\n> subset). If the effect only survives under ONE convention, it's usually noise.\n>\n> **Bottom line:** Claim B is rejected. After hardening no single MD feature\n> robustly beats GBSA. **Docking → GBSA → MD hierarchy is flat**, all CIs overlap.

In [ ]:
# --- notebook preamble ---
NB_STEM = "42_single_feature_bedroc"

import sys, os, json
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE
from discovery9.paths import ROOT, DERIVED, FIGURES, GBSA_STUDY
from discovery9.io import load_features
from discovery9.metrics import bedroc
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture


# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes x {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 1. Setup — why single-feature BEDROC is a useful diagnostic

BEDROC α=20 is our confirmatory early-enrichment metric — it heavily upweights the top of the ranked list. At α=20, the effective depth that matters is roughly `20/N ≈ 0.67` in normalised-rank space: only the top ~1.5 positions of a 30-ligand panel contribute meaningfully. So it's very sensitive to which molecule lands at rank 1 or 2, and nearly blind to what happens further down. Excellent as a diagnostic for a screening feature. Fragile as a foundation for a claim: small changes to the labelled set or scoring convention can move it 0.1–0.2 units.

Single-feature BEDROC asks the cleanest question: does this one number, used directly as a rank score with a fixed sign, put the actives on top for the panel? No fitting, no CV, no feature combination. The kind of test that either survives adversarial pressure or falls.

The panel here is the 8-target GBSA-scored subset (240 labelled complexes; 4A5S isn't rescored by upstream GBSA under the locked combo). GBSA-locked panel-mean BEDROC on this 8T subset (combo `igb2_di4_salt0.15_st0.0072`): **0.6088**. The canonical cross-notebook baseline is **0.541** on the full 9-target panel with 4A5S imputed at 0 — see `data/derived/canonical_baselines.csv`.

## 2. The naive result — reproducing the original headline

For every candidate MD feature and ligand-chem descriptor we compute per-target BEDROC α=20, take the panel mean, pick the sign (higher-is-better vs lower-is-better) that maximises the mean, and rank. Naive version — exactly what the original notebook did. No confound control, no re-labelling sensitivity.

In [ ]:
# Rebuild the naive per-feature BEDROC table exactly as in the original claim.
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from discovery9.io import load_features
from discovery9.metrics import bedroc
from discovery9.paths import GBSA_STUDY
from discovery9.style import NAVY, GOLD, GREY, GREY_DASH as GREYD

df = load_features(with_ligand_chem=True)
gbsa_raw = pd.read_csv(GBSA_STUDY / 'data' / 'raw' / 'gbsa_dG_raw.csv').rename(columns={'mean_dG_kcalmol':'gbsa_dG'})
LOCKED = 'igb2_di4_salt0.15_st0.0072'
gLK = gbsa_raw[gbsa_raw.combo == LOCKED][['complex_id','target','gbsa_dG']]

work = df.merge(gLK, on=['complex_id','target'], how='left').dropna(subset=['is_active'])
work = work[work.target != '4A5S']
print(f'labelled panel: {len(work)} complexes, {work.target.nunique()} targets')

# GBSA-locked panel BEDROC baseline (per target then mean).
baseline_gbsa = {t: bedroc(-g.gbsa_dG.values, g.is_active.astype(int).values)
                 for t, g in work.groupby('target')}
gbsa_mean = float(np.nanmean(list(baseline_gbsa.values())))
print(f'GBSA-locked baseline (panel mean BEDROC α=20): {gbsa_mean:.4f}')

MD_FEATS_ALL = [
    'rmsd_bb_mean_A','rmsd_bb_std_A','rmsd_as_bb_mean_A','rmsd_as_bb_std_A',
    'protein_rg_mean_A','protein_rg_std_A','as_ca_rmsf_mean_A','as_ca_rmsf_max_A',
    'lig_drift_mean_A','lig_drift_std_A','lig_drift_last_A',
    'lig_com_disp_mean_A','lig_com_disp_max_A','lig_com_disp_last_A','lig_escape_frac',
    'lig_internal_rmsd_mean_A','lig_internal_rmsd_std_A','lig_internal_rmsd_last_A',
    'lig_rmsf_mean_A','lig_rmsf_max_A','lig_buried_sasa_mean_A2','lig_buried_sasa_std_A2',
    'vdw_contacts_mean','vdw_contacts_std','n_hb_mean','n_hb_std','hb_persistence_frac',
    'salt_bridges_lp_mean','ifp_tanimoto_median_vs_ref','ifp_tanimoto_last_vs_ref','ifp_tanimoto_entropy',
    'lig_binding_modes_1A','lig_binding_modes_2A','lig_orient_autocorr_mean','lig_orient_autocorr_last',
    'lig_rg_mean_A','lig_asphericity_mean','lig_dipole_mean_eA','lig_dipole_std_eA',
    'coulomb_mean_arb','coulomb_std_arb',
]
LIG_FEATS_ALL = [
    'lig_MW','lig_n_heavy','lig_rot_bonds','lig_HBD','lig_HBA','lig_all_rings',
    'lig_LogP','lig_TPSA','lig_fraction_sp3','lig_partial_q_abs_sum',
]
ALL_FEATS = [f for f in (MD_FEATS_ALL + LIG_FEATS_ALL) if f in work.columns]

rows = []
for f in ALL_FEATS:
    per_t_pos, per_t_neg = {}, {}
    for t, g in work.groupby('target'):
        lab = g.is_active.astype(int).values
        vals = g[f].values
        per_t_pos[t] = bedroc(+vals, lab)
        per_t_neg[t] = bedroc(-vals, lab)
    mean_pos = float(np.nanmean(list(per_t_pos.values())))
    mean_neg = float(np.nanmean(list(per_t_neg.values())))
    direction = '+' if mean_pos >= mean_neg else '-'
    panel = max(mean_pos, mean_neg)
    rows.append({'feature': f, 'direction': direction, 'panel_bedroc': panel})
single = pd.DataFrame(rows).sort_values('panel_bedroc', ascending=False).reset_index(drop=True)

print('\nTop-15 single features (naive panel BEDROC α=20):')
print(single.head(15).round(4).to_string(index=False))

hit = single[single.feature == 'lig_buried_sasa_std_A2'].iloc[0]
print(f"\nlig_buried_sasa_std_A2 (direction {hit.direction}): {hit.panel_bedroc:.4f}"
      f"  vs GBSA-locked baseline {gbsa_mean:.4f}  (Δ = {hit.panel_bedroc - gbsa_mean:+.4f})")


In [ ]:
# Bar chart: top-15 single features, GOLD = top hit, NAVY = other features that beat 0.5.
top15 = single.head(15).iloc[::-1]
colors = []
for f, v in zip(top15.feature, top15.panel_bedroc):
    if f == 'lig_buried_sasa_std_A2':
        colors.append(GOLD)
    elif v >= 0.5:
        colors.append(NAVY)
    else:
        colors.append(GREYD)

fig, ax = plt.subplots(figsize=(9.5, 6.5))
ax.barh(range(len(top15)), top15.panel_bedroc, color=colors, edgecolor=NAVY, linewidth=0.5)
ax.axvline(gbsa_mean, color=GREYD, ls='--', lw=1.5, label=f'GBSA-locked baseline = {gbsa_mean:.3f}')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels([f'{f} ({d})' for f, d in zip(top15.feature, top15.direction)], fontsize=9)
ax.set_xlabel('naive panel BEDROC α=20 (single-feature ranker)')
ax.set_title('Top-15 single features — naive comparison against GBSA-locked')
ax.legend(fontsize=9, loc='lower right')
ax.set_axisbelow(True)
ax.xaxis.grid(True, color=GREY, alpha=0.5)
ax.set_xlim(0, max(0.75, float(top15.panel_bedroc.max()) + 0.05))
plt.tight_layout()

# Register for later export.
# (fig auto-captured by preamble hook)


## 3. Why the naive result is not credible — four independent attacks

The 0.674 vs 0.609 gap (both on the 8T subset; canonical 9T gap 0.603 vs 0.541) looked exciting. Review turned up four independent problems, each of which alone would kill a screening claim:

1. **Non-reproducibility across pipelines.** The naive BEDROC of `lig_buried_sasa_std_A2` is very sensitive to how ties are broken, how score direction is oriented, and how the labelled subset boundary is handled. Re-implementations in two independent scoring pipelines produced values from 0.61 to 0.71 for what should be the same number. A claim that moves 0.1 under re-implementation is not a claim, it's noise.

2. **Molecular-weight confounder.** The 30 ligands per target are not size-matched. `lig_buried_sasa_std_A2` correlates with MW (bigger ligand → bigger buried surface → wider absolute fluctuation), and in this panel MW is slightly enriched among actives. Residualise `lig_buried_sasa_std_A2` on `lig_MW` (per-target linear regression, take the residual as the new score) and the metric collapses to 0.506 — below GBSA-locked.

3. **Bound-only collapse.** Roughly half the trajectories still had the ligand bound at 30 ns; the other half showed at least partial dissociation. Restricting to the still-bound subset (`bound_only`) — which is what a screening use case would actually run on — drops the metric to 0.463. In the bound-only regime the signal is worse than random ranking.

4. **Decoy-label incoherence.** Decoys are defined by an activity threshold, not by structural or chemical dissimilarity. A meaningful fraction of the nominal decoys have pchembl values close to the active/decoy boundary — they may be weak binders mislabelled as decoys. That inflates variance and makes the exact 0.674 value depend on the arbitrary pchembl cutoff.

The fix-pack rebuilt the analysis under each of these lenses with proper bootstrap CIs and permutation p-values (bootstrap B=5000, permutation B=5000). Results below.

## 4. The hardened comparison — the six-condition table

`data/derived/hardened_claim_b.csv` holds the honest numbers. For each condition we report BEDROC, GBSA-locked BEDROC on the same subset (the baseline the feature has to beat on that subset), the delta, a 95% bootstrap CI on the delta, and a permutation p on the delta.

Conditions encode the four attacks:
- `naive` — no correction (the original claim).
- `mw_resid` — score residualised on `lig_MW` per target.
- `size_resid` — score residualised on `n_ligand_atoms` per target.
- `bound_only` — restrict to complexes still bound at 30 ns.
- `alt_label_7` — actives are pchembl ≥ 7, decoys pchembl ≤ 5 (tight-label variant).
- `alt_label_6` — actives are pchembl ≥ 6, decoys pchembl ≤ 5 (loose variant).

The three rows worth watching: the top hit `lig_buried_sasa_std_A2`, plus two ligand-chem sanity baselines `lig_MW` and `lig_TPSA` (physical size / polarity — features that could explain a spurious signal on their own).

## 4a. Docking → GBSA → MD hierarchy on the naive panel

Claim B originally asked "does MD beat GBSA?" without asking whether GBSA beats docking. The three-bar comparison below shows naive-panel BEDROC of (i) docking score (pose seed, GREY), (ii) GBSA-locked (NAVY, baseline-of-record), and (iii) the best single MD feature (`lig_buried_sasa_std_A2`, GOLD). Error bars are 95% bootstrap CI on per-target BEDROC (resample targets with replacement, B=5000, seed 20250901).

All three bars overlap within CI: the docking → GBSA → MD hierarchy is flat. The stepwise-physics story is not supported on this panel — the +0.10 gap docking → GBSA and the +0.06 gap GBSA → MD both sit well inside the CI of every bar.

In [ ]:
# iter-3 FIX 2: three-bar hierarchy (docking / GBSA-locked / best MD feature).
# Reads the hardened CSV so the numbers stay in sync with `reproduce/hardened_claim_b.py`.
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from discovery9.io import load_features, load_metadata, load_gbsa
from discovery9.metrics import bedroc
from discovery9.paths import DERIVED
from discovery9.style import NAVY, GOLD, GREY, GREY_DASH as GREYD

LOCKED = 'igb2_di4_salt0.15_st0.0072'
BEST_MD_FEAT = 'lig_buried_sasa_std_A2'
BOOT_B = 5000
RNG_SEED = 20250901
ALPHA = 20.0

meta = load_metadata()
feat = load_features(with_ligand_chem=True)
gbsa = load_gbsa(combo=LOCKED)[['complex_id','target','gbsa_dG']]

# Recover 4A5S is_active from metadata (iter-3 FIX 8): features.parquet was joined against
# MANIFEST.tsv, which has EMPTY is_active for the 4A5S rows.
lab = feat.merge(meta[['complex_id','target','is_active','pchembl','docking_score']],
                  on=['complex_id','target'], how='left', suffixes=('','_meta'))
if 'is_active_meta' in lab.columns:
    lab['is_active'] = (lab['is_active'].astype('boolean')
                        .combine_first(lab['is_active_meta'].astype('boolean')))
lab = lab.merge(gbsa, on=['complex_id','target'], how='left')
lab = lab.dropna(subset=['is_active']).copy()
lab['is_active'] = lab['is_active'].astype(bool).astype(int)
print(f'labelled panel (post-recovery): {len(lab)} rows, {lab.target.nunique()} targets: '
      f'{sorted(lab.target.unique())}')

# Per-target BEDROC, then panel mean + bootstrap CI on the mean over targets.
def per_target_and_boot(subset, score_col, sign, label_col='is_active', B=BOOT_B, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    tgts = sorted(subset[target_col := 'target'].unique())
    per_t = []
    for t in tgts:
        g = subset[subset.target == t]
        y = g[label_col].astype(int).to_numpy()
        s = sign * g[score_col].astype(float).to_numpy()
        b = bedroc(s, y, alpha=ALPHA)
        if np.isfinite(b):
            per_t.append(b)
    per_t = np.array(per_t, dtype=float)
    if len(per_t) < 2:
        return float('nan'), (float('nan'), float('nan')), per_t
    idx = rng.integers(0, len(per_t), size=(B, len(per_t)))
    boots = per_t[idx].mean(axis=1)
    return float(per_t.mean()), (float(np.percentile(boots, 2.5)),
                                 float(np.percentile(boots, 97.5))), per_t

# All three bars are computed on the SAME per-bar labelled subset (each ranker uses whatever
# rows have both a valid label and a valid score for its own column). We report N-targets in
# the annotation.
dock_sub = lab.dropna(subset=['docking_score'])
gbsa_sub = lab.dropna(subset=['gbsa_dG'])
md_sub   = lab.dropna(subset=[BEST_MD_FEAT])

dock_mean, dock_ci, dock_per_t = per_target_and_boot(dock_sub, 'docking_score', sign=-1)
gbsa_mean, gbsa_ci, gbsa_per_t = per_target_and_boot(gbsa_sub, 'gbsa_dG',       sign=-1)
md_mean,   md_ci,   md_per_t   = per_target_and_boot(md_sub,   BEST_MD_FEAT,    sign=+1)

for name, m, ci, nt in [('docking', dock_mean, dock_ci, len(dock_per_t)),
                        ('GBSA-locked', gbsa_mean, gbsa_ci, len(gbsa_per_t)),
                        (f'MD ({BEST_MD_FEAT})', md_mean, md_ci, len(md_per_t))]:
    print(f'  {name:38s} panel BEDROC = {m:.3f}  95% CI [{ci[0]:+.3f}, {ci[1]:+.3f}]   n_targets={nt}')

# 3-bar chart
fig, ax = plt.subplots(figsize=(7.5, 5.2))
labels = ['docking\n(pose seed)', f'GBSA-locked\n({LOCKED[:14]}…)',
          f'best MD feature\n({BEST_MD_FEAT})']
means  = [dock_mean, gbsa_mean, md_mean]
cis    = [dock_ci,   gbsa_ci,   md_ci]
n_targets_each = [len(dock_per_t), len(gbsa_per_t), len(md_per_t)]
colors = [GREY, NAVY, GOLD]
edges  = [NAVY, NAVY, NAVY]

err_lo = [max(0, m - lo) for m, (lo, hi) in zip(means, cis)]
err_hi = [max(0, hi - m) for m, (lo, hi) in zip(means, cis)]

x = np.arange(len(labels))
ax.bar(x, means, width=0.55, color=colors, edgecolor=edges, linewidth=1.0,
       yerr=[err_lo, err_hi], capsize=6, ecolor=GREYD)
for xi, m, ci, nt in zip(x, means, cis, n_targets_each):
    ax.text(xi, m + max(0, ci[1] - m) + 0.03, f'{m:.3f}\n[{ci[0]:.2f}, {ci[1]:.2f}]\nn_tgt={nt}',
            ha='center', va='bottom', fontsize=9, color=NAVY)
ax.axhline(0.5, color=GREY, ls=':', lw=1)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('panel BEDROC α=20')
ax.set_ylim(0, 1.0)
ax.set_title('Docking → GBSA → MD hierarchy on the naive panel\n'
             '(all three overlap within 95% CI — no stepwise physics improvement)')
ax.set_axisbelow(True); ax.yaxis.grid(True, color=GREY, alpha=0.5)
plt.tight_layout()

# (fig auto-captured by preamble hook)
print('\nCaption: The docking → GBSA → MD hierarchy is flat: all three overlap within CI.')

In [ ]:
# Load the hardened table and format the headline rows.
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from discovery9.paths import DERIVED
from discovery9.style import NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM

hard = pd.read_csv(DERIVED / 'hardened_claim_b.csv')

# iter-2 FIX 3: include pbc_clean (NEW), and keep the label variants for full picture
COND_ORDER = ['naive', 'mw_resid', 'size_resid', 'bound_only', 'pbc_clean', 'alt_label_7', 'alt_label_6']
FEATS_SHOW = ['lig_buried_sasa_std_A2', 'lig_MW', 'lig_TPSA']

sub = hard[hard.feature.isin(FEATS_SHOW)].copy()
sub['condition'] = pd.Categorical(sub.condition, categories=COND_ORDER, ordered=True)
sub['feature']   = pd.Categorical(sub.feature,   categories=FEATS_SHOW,  ordered=True)
sub = sub.sort_values(['feature', 'condition']).reset_index(drop=True)

sub_show = sub[['feature','condition','BEDROC','gbsa_BEDROC_on_subset','delta_vs_gbsa',
                'ci_lo','ci_hi','perm_p','n_actives','n_decoys']].round(4)
print('Hardened Claim B — seven-condition table for the top hit + two size/polarity baselines:')
print(sub_show.to_string(index=False))

# ---------- Bar plot: lig_buried_sasa_std_A2 across the 7 conditions, error bars = 95% CI on Delta. ----------
top = sub[sub.feature == 'lig_buried_sasa_std_A2'].copy()

vals = top.BEDROC.values
gbsa_on_subset = top.gbsa_BEDROC_on_subset.values
ci_lo_delta = top.ci_lo.values
ci_hi_delta = top.ci_hi.values
err_lo = np.clip(top.delta_vs_gbsa.values - ci_lo_delta, 0, None)
err_hi = np.clip(ci_hi_delta - top.delta_vs_gbsa.values, 0, None)

fig, ax = plt.subplots(figsize=(11, 5.6))
x = np.arange(len(top))

# iter-2 FIX 3: alt_label_7 is a DEGENERATE-PANEL artifact (3/8 targets have zero decoys under
# the strict filter and are silently NaN-dropped by panel BEDROC). Shade it in GREY, not GOLD,
# and add an explicit callout. All other bars use NAVY (no MD feature robustly beats GBSA).
bar_colors = []
for cond in top.condition.astype(str).tolist():
    if cond == 'alt_label_7':
        bar_colors.append(GREY)
    else:
        bar_colors.append(NAVY)

ax.bar(x, vals, width=0.55, color=bar_colors, edgecolor=NAVY,
       yerr=[err_lo, err_hi], capsize=5, ecolor=GREYD)

# GBSA baseline per subset - short tick per bar, and the naive baseline as a dashed line.
for xi, g in zip(x, gbsa_on_subset):
    ax.hlines(g, xi - 0.28, xi + 0.28, color=GREYD, lw=1.4)
gbsa_naive = float(top.iloc[0].gbsa_BEDROC_on_subset)
ax.axhline(gbsa_naive, color=GREYD, ls='--', lw=1.2,
           label=f'GBSA-locked (naive panel) = {gbsa_naive:.3f}')

# Annotate perm p on each bar.
for xi, v, p in zip(x, vals, top.perm_p.values):
    ax.text(xi, v + 0.02, f'p={p:.3f}', ha='center', va='bottom', fontsize=9, color=NAVY)

# iter-2 FIX 3: explicit degeneracy callout for the alt_label_7 bar.
try:
    alt7_i = list(top.condition.astype(str)).index('alt_label_7')
    ax.annotate('degenerate:\n3 targets NaN-dropped',
                xy=(alt7_i, vals[alt7_i]), xytext=(alt7_i, vals[alt7_i] + 0.20),
                ha='center', va='bottom', fontsize=9, color='#a02020',
                arrowprops=dict(arrowstyle='->', color='#a02020', lw=1.0))
except ValueError:
    pass

ax.set_xticks(x)
ax.set_xticklabels(top.condition.astype(str), rotation=15)
ax.set_ylim(0, 1.25)
ax.set_ylabel('BEDROC alpha=20')
ax.set_title('lig_buried_sasa_std_A2 across the 7 hardening conditions\n'
             '(GREY = degenerate panel; NAVY = other conditions; grey ticks = GBSA on subset)')
ax.legend(fontsize=9, loc='upper left')
ax.set_axisbelow(True)
ax.yaxis.grid(True, color=GREY, alpha=0.5)
plt.tight_layout()

# Register for later export.
# (fig auto-captured by preamble hook)

# iter-3 FIX 2 caption note: alt_label_7 is not feature-specific — the docking baseline
# also jumps from ~0.52 to ~0.81 under the same tight-label filter (see the docking_score
# rows of hardened_claim_b.csv). The filter creates an easier ranking problem for any
# monotone ranker, so we cannot claim the alt_label_7 signal is unique to MD features.
print('\nCaption note (iter-3 FIX 2): Not feature-specific: docking_score also jumps to 0.813')
print('under this filter — the alt-label creates an easier problem for any monotone ranker.')
try:
    hard_full = pd.read_csv(DERIVED / 'hardened_claim_b.csv')
    tri = hard_full[(hard_full.feature.isin(['docking_score','gbsa_dG',BEST_MD_FEAT])) &
                    (hard_full.condition.isin(['naive','alt_label_7']))]                    .pivot(index='feature', columns='condition', values='BEDROC')
    print('\nnaive vs alt_label_7 for the three rankers (from hardened_claim_b.csv):')
    print(tri.round(3).to_string())
except Exception as _e:
    print(f'(could not print alt_label_7 comparison table: {_e})')

## 5. The one caveat that survives — the tight-label condition

Under `alt_label_7` (actives pchembl ≥ 7, decoys pchembl ≤ 5, middling ligands 5 < pchembl < 7 dropped), `lig_buried_sasa_std_A2` reaches BEDROC = 0.978 with permutation p = 0.015. That is a real p<0.05 signal.

Read it soberly:
- Sample size is very small — ~84 actives, ~30 decoys across 8 targets, ~10 per target after tightening.
- 95% bootstrap CI on Δ is [−0.084, +0.815]. Lower bound touches zero, upper bound is huge — the signal is real but its size is very uncertain.
- Comparison is against a different GBSA baseline (0.727 on the same tight-label subset, not the 0.609 naive-8T-panel baseline / canonical 9T = 0.541 from the original claim). Comparing 0.978 to 0.727 is fair; comparing 0.978 to 0.609 is not.
- No external test set. Cross-validating on the same 8 targets and 30 ligands the ranking was built from double-counts the signal.

The cell below prints per-target labelled counts under the tight-label filter, so the sample-size caveat is auditable.

In [ ]:
# Per-target active/decoy counts under the alt_label_7 filter (pchembl >= 7 vs pchembl <= 5).
import numpy as np, pandas as pd
from discovery9.io import load_features
from discovery9.paths import GBSA_STUDY

df = load_features(with_ligand_chem=True)
gbsa_raw = pd.read_csv(GBSA_STUDY / 'data' / 'raw' / 'gbsa_dG_raw.csv').rename(columns={'mean_dG_kcalmol':'gbsa_dG'})
LOCKED = 'igb2_di4_salt0.15_st0.0072'
gLK = gbsa_raw[gbsa_raw.combo == LOCKED][['complex_id','target','gbsa_dG']]
work = df.merge(gLK, on=['complex_id','target'], how='left').dropna(subset=['is_active'])
work = work[work.target != '4A5S']

tight = work.dropna(subset=['pchembl']).copy()
tight['label_alt7'] = np.where(tight.pchembl >= 7.0, 1,
                       np.where(tight.pchembl <= 5.0, 0, np.nan))
tight = tight.dropna(subset=['label_alt7'])

cnt = (tight.groupby('target')['label_alt7']
            .agg(n_actives=lambda s: int((s == 1).sum()),
                 n_decoys =lambda s: int((s == 0).sum()))
            .reset_index())
cnt['n_total'] = cnt.n_actives + cnt.n_decoys
print('Per-target counts under the tight-label filter (pchembl >= 7 = active, <= 5 = decoy):')
print(cnt.to_string(index=False))
print(f'\nPanel totals under alt_label_7: n_actives={cnt.n_actives.sum()}, n_decoys={cnt.n_decoys.sum()}, targets={len(cnt)}')
print('For comparison, the naive panel has 80 actives / 160 decoys across 8 targets (10/20 per target).')


## 6. Physical interpretation — honest

The original claim said a large `lig_buried_sasa_std_A2` is an induced-fit signature: the pocket rearranges around the active ligand, alternating between multiple stable buried configurations, so buried SASA fluctuates. That's one mechanism that would give wider buried-SASA over 30 ns.

Not the only one. Wider fluctuations fit equally well with:
- **Loose-pocket tumbling** — the ligand is bound but rotates or translates within a large, smooth pocket, moving in and out of contact with pocket walls.
- **Partial dissociation and re-binding** — the ligand transiently detaches from part of the pocket and comes back, giving bimodal buried-SASA over the trajectory.
- **Pocket collapse** — the pocket itself is unstable and closes/opens around a passive ligand.

From 30 ns simulations of a single starting pose we can't tell these apart — they all produce a similar summary statistic. An induced-fit interpretation would need much longer trajectories, an explicit re-docking test, or independent structural evidence for pocket rearrangement (e.g. pocket-lining RMSF, bound vs apo). None of that exists in the present data.

## 7. Verdict

> **Claim B is retracted.**
>
> - On the 8T subset the delta was +0.065 with a 95% CI [-0.22, +0.22]. That's zero as far as anyone can tell. Canonical 9T values (4A5S imputed at 0): feature 0.603 vs GBSA-locked 0.541.
> - After MW/size residualisation the score drops to 0.506, below GBSA. The naive value was a size confounder.
> - After excluding PBC-artefact trajectories it drops further.
> - The alt-label pchembl≥7 result (0.978) is a degenerate-panel artefact. Three of the 8 targets have zero decoys under the strict filter and get silently NaN-dropped. The "signal" is driven by one fully-evaluable target (8ELC), which is a congeneric within-series ranking, not target discrimination.
> - **No MD feature robustly beats canonical GBSA-locked (0.541 on 9T, 0.609 on 8T) on the discovery-9 panel.** The real finding is negative: no single MD feature we tested beats one locked GBSA combo on the panel.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
